# Solução Final — Escoragem do Test, Submissão Kaggle e Diagnóstico Sem Target

Este caderno executa a **entrega final** do sistema de detecção de fraudes sobre a base cega do Kaggle (`test_transaction.csv` e `test_identity.csv`), utilizando o artefato congelado exportado em `02_Modelagem.ipynb` (`LGBM_Otimizado` + **Calibração Isotônica** + **Threshold Dinâmico por Ticket**).

> **Por que exportar `prob_cal` na submissão Kaggle?** O LightGBM foi treinado com `scale_pos_weight` (~27:1), o que desloca a probabilidade bruta (`prob_raw`) para cima. A regressão isotônica ajusta essa saída em **`prob_cal` ∈ [0, 1]**, restaurando a probabilidade real calibrada de fraude — escala correta tanto para a métrica **ROC-AUC** da competição quanto para comparação direta com o threshold dinâmico (`pred_fraude` 0/1).

## Sumário

| Seção | Objetivo | Entregáveis / Análises |
|:---:|---|---|
| [**1.0**](#10-preparacao-do-ambiente-download-e-carga) | Preparação e Carga | Download via `kagglehub` (`test_*`), carga do artefato final e `df_master` |
| [**2.0**](#20-engenharia-de-atributos-e-paridade-com-o-treino) | Paridade de Pipeline | Normalização `id-` → `id_`, FE idêntica ao `01_EDA`, alinhamento `log1p` e encoders do treino |
| [**3.0**](#30-escoragem-e-calibracao-isotonica) | Inferência | Aplicação de `aplicar_regra_dinamica` (`prob_raw`, `prob_cal`, `threshold`, `pred_fraude`) |
| [**4.0**](#40-exportacao-das-submissoes) | Entrega (CSVs) | `data/submissions/submission_kaggle.csv` e `data/submissions/predicoes_test_binarias.csv` |
| [**5.0**](#50-analise-exploratoria-pos-escoragem-do-test-sem-target) | Diagnóstico do Test | Faixas de ticket, evolução mensal/semanal das predições, PSI (In-Time × Test) e `ProductCD` |
| [**6.0**](#60-sintese-executiva) | Consolidação | Resumo operacional de volumetria, exposição financeira e estabilidade |

## 1.0 Preparação do Ambiente, Download e Carga

Configuramos os diretórios do projeto, carregamos o artefato congelado (`artifacts/modelo_fraude_final.joblib`) e a base processada de referência (`df_master.parquet`), e garantimos que `test_transaction.csv` e `test_identity.csv` estejam disponíveis em `data/raw/` usando o fluxo do `01_EDA.ipynb`.

In [ ]:
# ==============================================================================
# 1.1 CONFIGURAÇÃO DO AMBIENTE, DIRETÓRIOS E CARGA DO ARTEFATO
# ==============================================================================
import gc
import os
import sys
import json
import warnings
import importlib
from pathlib import Path

warnings.filterwarnings('ignore')

# Resolução robusta da raiz do projeto (Local e Colab)
current_dir = Path(os.getcwd()).resolve()
if (current_dir / 'utils.py').exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / 'utils.py').exists():
    PROJECT_ROOT = current_dir.parent
elif (current_dir.parent.parent / 'utils.py').exists():
    PROJECT_ROOT = current_dir.parent.parent
else:
    PROJECT_ROOT = current_dir

os.chdir(PROJECT_ROOT)
for p in [str(PROJECT_ROOT), os.path.abspath('.'), os.path.abspath('..')]:
    if p not in sys.path:
        sys.path.append(p)

import joblib
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import utils
importlib.reload(utils)
from utils import (
    aplicar_estilo_azul,
    preparar_dados_teste_inferencia,
    aplicar_regra_dinamica,
    escorar_particao,
    adicionar_bloco_temporal,
    resumir_predicoes_test,
    plot_evolucao_predicoes_test,
    plot_predicoes_por_categoria,
    calcular_psi_scores,
    plot_psi_score_faixas,
    relatorio_psi_features,
    plot_comparativo_psi_features,
)

aplicar_estilo_azul()

# Diretórios oficiais
DIR_RAW  = PROJECT_ROOT / 'data' / 'raw'
DIR_PROC = PROJECT_ROOT / 'data' / 'processed'
DIR_SUB  = PROJECT_ROOT / 'data' / 'submissions'
DIR_ART  = PROJECT_ROOT / 'artifacts'
DIR_FIG  = PROJECT_ROOT / 'reports' / 'figures' / 'solucao'
DIR_MET  = PROJECT_ROOT / 'reports' / 'metrics'
for d in [DIR_RAW, DIR_SUB, DIR_FIG, DIR_MET]:
    d.mkdir(parents=True, exist_ok=True)

# Carga do artefato congelado e metadados de treino
artefato = joblib.load(DIR_ART / 'modelo_fraude_final.joblib')
FEATURES = list(artefato['features'])
regra_decisao = artefato['regra_decisao']

meta_path = DIR_PROC / 'features_metadata.json'
meta = json.loads(meta_path.read_text(encoding='utf-8')) if meta_path.exists() else None
df_master = pd.read_parquet(DIR_PROC / 'df_master.parquet')

print(f"Projeto          : {PROJECT_ROOT}")
print(f"Modelo congelado : {artefato.get('nome_modelo', 'LGBM_Otimizado')} | {len(FEATURES)} features")
print(f"Regra dinâmica   : t_min={regra_decisao['t_min']:.3f}, t_max={regra_decisao['t_max']:.3f}, k={regra_decisao['k']:.1f}")
print(f"df_master (ref.) : {df_master.shape[0]:,} linhas × {df_master.shape[1]} colunas")

In [ ]:
# ==============================================================================
# 1.2 DOWNLOAD (KAGGLEHUB) E LEITURA DAS BASES DE TEST
# ==============================================================================
FORCAR_DOWNLOAD = False  # Altere para True se desejar baixar novamente mesmo com cache local
arq_test_tr = DIR_RAW / 'test_transaction.csv'
arq_test_id = DIR_RAW / 'test_identity.csv'

if FORCAR_DOWNLOAD or not (arq_test_tr.exists() and arq_test_id.exists()):
    print("[KaggleHub] Autenticando e baixando arquivos da competição ieee-fraud-detection...")
    kagglehub.login()
    path_kaggle = kagglehub.competition_download(
        'ieee-fraud-detection',
        force_download=FORCAR_DOWNLOAD,
        output_dir=str(DIR_RAW),
    )
    print(f"Arquivos baixados em: {path_kaggle}")
else:
    print(f"[Cache local] Bases de teste encontradas em {DIR_RAW}.")

df_test_tr = pd.read_csv(arq_test_tr)
df_test_id = pd.read_csv(arq_test_id)

cols_hifen = [c for c in df_test_id.columns if c.startswith('id-')]
print(f"test_transaction : {df_test_tr.shape[0]:,} linhas × {df_test_tr.shape[1]} colunas | "
      f"TransactionID {df_test_tr['TransactionID'].min():,} → {df_test_tr['TransactionID'].max():,}")
print(f"test_identity    : {df_test_id.shape[0]:,} linhas × {df_test_id.shape[1]} colunas | "
      f"{len(cols_hifen)} colunas com hífen ('id-*') detectadas para normalização")

## 2.0 Engenharia de Atributos e Paridade com o Treino

O conjunto de teste do Kaggle traz duas armadilhas clássicas que invalidariam a inferência se tratadas de forma ingênua:
1. **`test_identity.csv` usa hífens (`id-01`..`id-38`)**, enquanto o treino usa sublinhados (`id_01`..`id_38`).
2. **Escala `log1p` e Encoders:** no `01_EDA`, variáveis assimétricas receberam `np.log1p`, e no `02_Modelagem` o `OrdinalEncoder`, a razão por `card1` e as frequências foram ajustados **exclusivamente sobre os 60% de treino**.

A função `preparar_dados_teste_inferencia` replica integralmente a engenharia de atributos do `01_EDA`, alinha a escala `log1p` por distância KS contra `df_master` e aplica as estatísticas e o encoder ajustados apenas no treino.

In [ ]:
# ==============================================================================
# 2.1 ENGENHARIA DE ATRIBUTOS, ALINHAMENTO LOG1P E PARIDADE COM PREPARAR_DADOS_MODELAGEM
# ==============================================================================
prep_test = preparar_dados_teste_inferencia(
    df_test_tr=df_test_tr,
    df_test_id=df_test_id,
    df_master=df_master,
    features_modelo=FEATURES,
    meta=meta,
    train_ratio=0.60,
    test_ratio=0.20,
)

X_test = prep_test['X_test']
amt_usd_test = prep_test['amt_usd']

# Libera tabelas brutas da memória após consolidação
del df_test_tr, df_test_id
gc.collect()

# Asserções de integridade pré-escoragem
assert list(X_test.columns) == FEATURES, "Ordem/conjunto de colunas de X_test difere do artefato."
assert not np.isnan(prep_test['TransactionID']).any(), "TransactionID contém valores nulos."
assert np.isfinite(amt_usd_test).all(), "amt_usd contém NaN ou infinito."

print(f"Colunas convertidas p/ log1p : {len(prep_test['cols_log1p'])} | Ex.: {prep_test['cols_log1p'][:8]}")
print(f"Período Test (dias)          : {prep_test['TransactionDT'].min()/86400:.1f} → {prep_test['TransactionDT'].max()/86400:.1f}")

## 3.0 Escoragem e Calibração Isotônica

Aplicamos `aplicar_regra_dinamica(artefato, X_test, amt_usd_test)` passando o valor **real em US\$** (antes de `log1p`). Assim obtemos:
- **`prob_raw`**: escore bruto do LightGBM (não calibrado, inflado pelo `scale_pos_weight`).
- **`prob_cal`**: probabilidade **corrigida pela regressão isotônica** em `[0, 1]` (**score oficial para a submissão Kaggle**).
- **`threshold_dinamico`**: ponto de corte adaptativo conforme o ticket da transação.
- **`pred_fraude`**: decisão binária operacional (`1` se `prob_cal >= threshold_dinamico`, caso contrário `0`).

In [ ]:
# ==============================================================================
# 3.1 ESCORAGEM DO TEST E REFERÊNCIAS IN-TIME (VALIDAÇÃO) / OOT
# ==============================================================================
res_test = aplicar_regra_dinamica(artefato, X_test, amt_usd_test)

df_pred = pd.DataFrame({
    'TransactionID': prep_test['TransactionID'],
    'TransactionDT': prep_test['TransactionDT'],
    'ProductCD': prep_test['ProductCD'],
    'amt_usd': amt_usd_test,
    'prob_raw': np.asarray(res_test['prob_raw'], dtype='float64'),
    'prob_cal': np.asarray(res_test['prob_cal'], dtype='float64'),
    'threshold_dinamico': np.asarray(res_test['threshold'], dtype='float64'),
    'pred_fraude': np.asarray(res_test['bloquear'], dtype='int8'),
})
df_pred['amt_bloqueado_usd'] = np.where(df_pred['pred_fraude'] == 1, df_pred['amt_usd'], 0.0)

# Referência de validação In-Time e OOT (fora do treino) para comparação de PSI e taxa base
df_val_scores = escorar_particao(artefato, prep_test['X_val'], prep_test['val_df'], 'Validação', 'In-Time')
df_oot_scores = escorar_particao(artefato, prep_test['X_oot'], prep_test['oot_df'], 'OOT', 'OOT')

resumo_escala = pd.DataFrame({
    'Escala': ['prob_raw (LightGBM bruto)', 'prob_cal (Regressão Isotônica)', 'threshold_dinamico'],
    'Mín': [df_pred['prob_raw'].min(), df_pred['prob_cal'].min(), df_pred['threshold_dinamico'].min()],
    'Mediana': [df_pred['prob_raw'].median(), df_pred['prob_cal'].median(), df_pred['threshold_dinamico'].median()],
    'Média': [df_pred['prob_raw'].mean(), df_pred['prob_cal'].mean(), df_pred['threshold_dinamico'].mean()],
    'P99': [df_pred['prob_raw'].quantile(0.99), df_pred['prob_cal'].quantile(0.99), df_pred['threshold_dinamico'].quantile(0.99)],
    'Máx': [df_pred['prob_raw'].max(), df_pred['prob_cal'].max(), df_pred['threshold_dinamico'].max()],
})
display(resumo_escala.style.format({'Mín': '{:.4f}', 'Mediana': '{:.4f}', 'Média': '{:.4f}', 'P99': '{:.4f}', 'Máx': '{:.4f}'})
        .set_caption('Distribuição dos Scores e Threshold Dinâmico no Test'))

print(f"Fraudes previstas no Test : {df_pred['pred_fraude'].sum():,} / {len(df_pred):,} "
      f"({df_pred['pred_fraude'].mean()*100:.2f}%) | Ref. Validação: {df_val_scores['bloquear'].mean()*100:.2f}% | "
      f"Ref. OOT: {df_oot_scores['bloquear'].mean()*100:.2f}%")

## 4.0 Exportação das Submissões

Exportamos os dois arquivos em `data/submissions/`:
1. **`submission_kaggle.csv`**: formato estrito exigido pelo Kaggle (`TransactionID,isFraud`), com `isFraud = prob_cal` na escala calibrada `[0, 1]`.
2. **`predicoes_test_binarias.csv`**: arquivo operacional completo com a decisão binária **`pred_fraude` (0/1)**, além de `prob_raw`, `prob_cal`, `threshold_dinamico`, `amt_usd`, `TransactionDT` e `ProductCD` para auditoria.

In [ ]:
# ==============================================================================
# 4.1 EXPORTAÇÃO E VALIDAÇÃO DOS DOIS CSVs EM data/submissions/
# ==============================================================================
arq_kaggle = DIR_SUB / 'submission_kaggle.csv'
arq_binario = DIR_SUB / 'predicoes_test_binarias.csv'

df_kaggle = pd.DataFrame({
    'TransactionID': df_pred['TransactionID'].astype('int64'),
    'isFraud': df_pred['prob_cal'].astype('float64'),
})

cols_auditoria = [
    'TransactionID', 'TransactionDT', 'ProductCD', 'amt_usd',
    'prob_raw', 'prob_cal', 'threshold_dinamico', 'pred_fraude',
]
df_binario = df_pred[cols_auditoria].copy()
df_binario['TransactionID'] = df_binario['TransactionID'].astype('int64')

df_kaggle.to_csv(arq_kaggle, index=False, float_format='%.6f')
df_binario.to_csv(arq_binario, index=False, float_format='%.6f')

# Validações automáticas de integridade pós-escrita
chk_kag = pd.read_csv(arq_kaggle)
chk_bin = pd.read_csv(arq_binario)
assert list(chk_kag.columns) == ['TransactionID', 'isFraud'], "Cabeçalho inválido na submissão Kaggle."
assert len(chk_kag) == len(df_pred) and len(chk_bin) == len(df_pred), "Divergência no nº de linhas exportadas."
assert not chk_kag.isna().any().any() and not chk_bin['pred_fraude'].isna().any(), "NaNs detectados nos CSVs."
assert chk_kag['isFraud'].between(0.0, 1.0).all(), "Scores fora do intervalo [0, 1]."
assert set(chk_bin['pred_fraude'].unique()).issubset({0, 1}), "pred_fraude deve conter apenas 0 e 1."

print(f"[OK] Kaggle CSV    : {arq_kaggle} ({arq_kaggle.stat().st_size / 1e6:.2f} MB)")
print(f"[OK] Binário CSV   : {arq_binario} ({arq_binario.stat().st_size / 1e6:.2f} MB)")
print("\nAmostra — submission_kaggle.csv:")
display(chk_kag.head())
print("\nAmostra — predicoes_test_binarias.csv:")
display(chk_bin.head())

## 5.0 Análise Exploratória Pós-Escoragem do Test (Sem Target)

Como a base de teste não possui rótulo real (`isFraud`), avaliamos a coerência das predições em quatro frentes enxutas:
1. **Valores Financeiros & Faixas de Ticket:** onde se concentra o volume bloqueado em US\$.
2. **Evolução Temporal:** estabilidade da taxa prevista e do `prob_cal` ao longo dos meses/semanas do Test.
3. **PSI (In-Time × Test):** deslocamento na distribuição de scores e nas features mais importantes.
4. **Perfil por `ProductCD`:** aderência das taxas previstas à hierarquia histórica de risco por canal.

### 5.1 Distribuição Financeira & Risco por Faixa de Ticket

In [ ]:
# ==============================================================================
# 5.1 VALORES TRANSACIONADOS E RISCO PREVISTO POR FAIXA DE TICKET
# ==============================================================================
vol_total = df_pred['amt_usd'].sum()
vol_bloq = df_pred['amt_bloqueado_usd'].sum()
tkt_aprov = df_pred.loc[df_pred['pred_fraude'] == 0, 'amt_usd'].median()
tkt_bloq = df_pred.loc[df_pred['pred_fraude'] == 1, 'amt_usd'].median()

print(f"Volume financeiro total (Test) : US$ {vol_total:,.2f}")
print(f"Volume retido (pred_fraude=1)  : US$ {vol_bloq:,.2f} ({vol_bloq/vol_total*100:.2f}% do GMV)")
print(f"Ticket mediano (Aprovadas)     : US$ {tkt_aprov:,.2f} | (Bloqueadas): US$ {tkt_bloq:,.2f}")

df_pred['Faixa_Ticket'] = pd.cut(
    df_pred['amt_usd'],
    bins=[0, 50, 150, 500, np.inf],
    include_lowest=True,
    labels=['Até $50', '$50–$150', '$150–$500', '>$500'],
)
df_faixas = resumir_predicoes_test(df_pred, 'Faixa_Ticket')

display(df_faixas[['Faixa_Ticket', 'Transacoes', '%_Base', 'Prev_Fraudes', 'Taxa_Prevista_%',
                   'Prob_Cal_Media_%', 'Threshold_Medio_%', 'Valor_Bloqueado_USD', '%_Exposicao_Bloq']]
        .style.format({
            'Transacoes': '{:,.0f}', '%_Base': '{:.1f}%', 'Prev_Fraudes': '{:,.0f}',
            'Taxa_Prevista_%': '{:.2f}%', 'Prob_Cal_Media_%': '{:.2f}%', 'Threshold_Medio_%': '{:.2f}%',
            'Valor_Bloqueado_USD': '${:,.2f}', '%_Exposicao_Bloq': '{:.1f}%',
        }).set_caption('Volumetria e Exposição Prevista por Faixa de Ticket (Test)'))

plot_predicoes_por_categoria(
    df_faixas, 'Faixa_Ticket',
    titulo='Test — Taxa Prevista e Exposição por Faixa de Ticket',
    subtitulo='Note o efeito do threshold dinâmico restringindo falsos positivos no topo e capturando volume médio/alto',
    filepath=DIR_FIG / '01_test_faixas_ticket.png',
)

### 5.2 Comportamento Temporal das Predições (Meses e Semanas do Test)

In [ ]:
# ==============================================================================
# 5.2 EVOLUÇÃO MENSAL E SEMANAL DAS PREDIÇÕES NO TEST
# ==============================================================================
dt_ini_test = float(df_pred['TransactionDT'].min())
df_pred_t = adicionar_bloco_temporal(df_pred, dt_inicio=dt_ini_test, dias_por_bloco=30, nome_coluna='mes_test')
df_pred_t['mes_test'] = df_pred_t['mes_test'] + 1
df_pred_t = adicionar_bloco_temporal(df_pred_t, dt_inicio=dt_ini_test, dias_por_bloco=7, nome_coluna='semana_test')
df_pred_t['semana_test'] = df_pred_t['semana_test'] + 1

df_mensal = resumir_predicoes_test(df_pred_t, 'mes_test')
df_semanal = resumir_predicoes_test(df_pred_t, 'semana_test')
# Remove eventual última semana parcial com poucas horas se necessário apenas para visualização limpa
df_semanal_plot = df_semanal[df_semanal['Transacoes'] >= 1000].copy()

display(df_mensal[['mes_test', 'Transacoes', '%_Base', 'Prev_Fraudes', 'Taxa_Prevista_%',
                   'Prob_Cal_Media_%', 'Valor_Total_USD', 'Valor_Bloqueado_USD']]
        .style.format({
            'Transacoes': '{:,.0f}', '%_Base': '{:.1f}%', 'Prev_Fraudes': '{:,.0f}',
            'Taxa_Prevista_%': '{:.2f}%', 'Prob_Cal_Media_%': '{:.2f}%',
            'Valor_Total_USD': '${:,.2f}', 'Valor_Bloqueado_USD': '${:,.2f}',
        }).set_caption('Comportamento Mensal do Target Previsto no Test (Meses de 30 dias)'))

plot_evolucao_predicoes_test(
    df_semanal_plot, col_bloco='semana_test', rotulo_bloco='Semana do Test',
    filepath=DIR_FIG / '02_test_evolucao_semanal.png',
)

### 5.3 Estabilidade e Drift (PSI: In-Time vs. Test)

Comparamos a distribuição dos escores calibrados (`Validação In-Time` × `Test`) e das `75` features do modelo (`In-Time` × `Test`). Pela convenção regulatória: **Estável (< 0,10)** · **Alerta (0,10–0,25)** · **Crítico (> 0,25)**.

In [ ]:
# ==============================================================================
# 5.3 PSI DO SCORE E FEATURE DRIFT (IN-TIME VS. TEST)
# ==============================================================================
df_psi_score, psi_score_total = calcular_psi_scores(df_val_scores['prob_cal'], df_pred['prob_cal'], bins=10)
plot_psi_score_faixas(
    df_psi_score, psi_score_total, rotulo_atual='Test',
    filepath=DIR_FIG / '03_test_psi_score.png',
)

df_psi_feats = relatorio_psi_features(prep_test['X_in_time'], X_test, features=FEATURES, bins=10)
cont_status = df_psi_feats['Status'].value_counts().to_dict()
print(f"Status das {len(df_psi_feats)} features (In-Time → Test): {cont_status}")

display(df_psi_feats.head(15).style.format({
    'PSI': '{:.4f}', 'KS_shift': '{:.4f}',
    '%_Nulos_InTime': '{:.1f}%', '%_Nulos_OOT': '{:.1f}%',
}).set_caption('Top 15 Features por PSI (In-Time → Test · coluna %_Nulos_OOT representa Test)'))

top_feats_psi = df_psi_feats['Feature'].head(4).tolist()
plot_comparativo_psi_features(
    prep_test['X_in_time'], X_test, features=top_feats_psi,
    rotulo_atual='Test', filepath=DIR_FIG / '04_test_psi_features.png',
)

### 5.4 Perfil das Predições por `ProductCD`

In [ ]:
# ==============================================================================
# 5.4 SEGMENTAÇÃO POR PRODUCTCD (W, C, R, H, S)
# ==============================================================================
df_prod = resumir_predicoes_test(df_pred, 'ProductCD').sort_values('Taxa_Prevista_%', ascending=False).reset_index(drop=True)

display(df_prod[['ProductCD', 'Transacoes', '%_Base', 'Prev_Fraudes', 'Taxa_Prevista_%',
                 'Prob_Cal_Media_%', 'Ticket_Medio_USD', 'Valor_Bloqueado_USD', '%_Exposicao_Bloq']]
        .style.format({
            'Transacoes': '{:,.0f}', '%_Base': '{:.1f}%', 'Prev_Fraudes': '{:,.0f}',
            'Taxa_Prevista_%': '{:.2f}%', 'Prob_Cal_Media_%': '{:.2f}%',
            'Ticket_Medio_USD': '${:,.2f}', 'Valor_Bloqueado_USD': '${:,.2f}', '%_Exposicao_Bloq': '{:.1f}%',
        }).set_caption('Predições e Exposição no Test por ProductCD'))

plot_predicoes_por_categoria(
    df_prod, 'ProductCD',
    titulo='Test — Taxa de Fraude Prevista e Exposição por ProductCD',
    subtitulo='A hierarquia entre canais digitais (C) e tradicionais (W) permanece coerente com o EDA',
    filepath=DIR_FIG / '05_test_product_cd.png',
)

## 6.0 Síntese Executiva

Consolidamos os indicadores-chave da escoragem cega do Test e exportamos um resumo estruturado em `reports/metrics/test_resumo_escoragem.json`.

In [ ]:
# ==============================================================================
# 6.1 CONSOLIDAÇÃO EXECUTIVA E EXPORTAÇÃO DO RESUMO JSON
# ==============================================================================
resumo_executivo = {
    'n_transacoes_test': int(len(df_pred)),
    'n_fraudes_previstas': int(df_pred['pred_fraude'].sum()),
    'taxa_fraude_prevista_pct': round(float(df_pred['pred_fraude'].mean() * 100), 4),
    'prob_cal_media_pct': round(float(df_pred['prob_cal'].mean() * 100), 4),
    'volume_total_usd': round(float(vol_total), 2),
    'volume_bloqueado_usd': round(float(vol_bloq), 2),
    'pct_volume_bloqueado': round(float(vol_bloq / vol_total * 100), 4),
    'psi_score_val_vs_test': round(float(psi_score_total), 5),
    'features_psi_critico': int((df_psi_feats['Status'] == 'Crítico').sum()),
    'features_psi_alerta': int((df_psi_feats['Status'] == 'Alerta').sum()),
    'arquivo_kaggle': str(arq_kaggle.relative_to(PROJECT_ROOT)),
    'arquivo_binario': str(arq_binario.relative_to(PROJECT_ROOT)),
}

arq_json = DIR_MET / 'test_resumo_escoragem.json'
arq_json.write_text(json.dumps(resumo_executivo, indent=2, ensure_ascii=False), encoding='utf-8')

display(pd.DataFrame([resumo_executivo]).T.rename(columns={0: 'Valor'}))
print(f"\nResumo salvo em: {arq_json}")